# LandingAI ADE — Accident Statement Parser & Debug Visualizer

## Overview
This notebook parses the `AccidentStatement.pdf` using **LandingAI ADE** (`dpt-2-latest`),
a vision-language model designed for complex document layouts including nested tables and forms.

It was built as an alternative to Databricks `ai_parse_document`, which struggles with
the nested table structure of the European Accident Statement form.

| File | Format | Key challenge |
|---|---|---|
| `AccidentStatement.pdf` | PDF (form / tables) | Nested tables, bilingual fields, checkboxes |

## Notebook structure
| Section | Description |
|---|---|
| **1. Configuration** | Unity Catalog paths, file registry |
| **2. Parse** | LandingAI ADE API call, chunk extraction |
| **3. Inspect** | Raw chunk dump for debugging |
| **4. Visualize** | Side-by-side page image vs rendered markdown |

## Output
Each parsed chunk contains:
- `chunk.type` — element type (`text`, `table`, `marginalia`, `figure`, …)
- `chunk.markdown` — content as Markdown/HTML (tables rendered as `<table>`)
- `chunk.grounding` — bounding box + page number for spatial reference


In [0]:
%pip install -q pdf2image==1.17.0 Pillow==10.3.0 numpy==1.23.5 scipy==1.11.1  landingai-ade==1.12.0 pymupdf markdown --quiet
dbutils.library.restartPython() 

## 1. Configuration

Load the shared Unity Catalog parameters (`catalog`, `schema`) from the central config notebook,
then define the volume paths and the list of files to process.

In [0]:
%run ../_config/config_unity_catalog

In [0]:
import json
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ── Paths ──────────────────────────────────────────────────────────────────
PATH_VOLUME   = f"/Volumes/{catalog}/{schema}/raw_data/multiformat"
PATH_IMAGES   = f"/Volumes/{catalog}/{schema}/raw_data/parsed_images"
PATH_RESULTS  = f"/Volumes/{catalog}/{schema}/raw_data/parse_results"

# Create output directories if they don't exist
dbutils.fs.mkdirs(PATH_IMAGES)
dbutils.fs.mkdirs(PATH_RESULTS)

# ── Files to analyse ───────────────────────────────────────────────────────
FILES = {
    "AccidentStatement_pdf" : "AccidentStatement.pdf",
}

print(f"Volume path   : {PATH_VOLUME}")
print(f"Images output : {PATH_IMAGES}")
print(f"Results output: {PATH_RESULTS}")
print(f"Files to parse: {list(FILES.values())}")

## 2. Parse with LandingAI ADE

`dpt-2-latest` is LandingAI's vision-language model optimised for dense document layouts.
It returns a list of **chunks** — spatial segments of the page, each with:
- a type (`text`, `table`, `figure`, `marginalia`, …)
- a markdown representation of the content
- a grounding bounding box (normalised coordinates, page index)

The API key is read from Databricks secrets to avoid hardcoding credentials.
`save_to` persists the raw grounding JSON to the UC volume for auditability.



In [0]:
import os, time
from pathlib import Path
from landingai_ade import LandingAIADE
SOURCE_PDF       = f"{PATH_VOLUME}/{FILES['AccidentStatement_pdf']}"
# -- API key from Databricks secrets (recommended) or env var
try:
    LANDING_AI_KEY = dbutils.secrets.get(scope='landing_ai', key='vision_agent_api_key')
except Exception:
    LANDING_AI_KEY = os.environ.get('vision_agent_api_key', 'YOUR_API_KEY_HERE')

client = LandingAIADE(apikey=LANDING_AI_KEY)
GROUNDING_DIR = f"{PATH_RESULTS}/landingai_groundings"
dbutils.fs.mkdirs(GROUNDING_DIR)

# -- Parse preprocessed PDF
print('Parsing with LandingAI ADE (dpt-2-latest) ...')
t0 = time.time()
ade_response = client.parse(
    document  = Path(SOURCE_PDF),
    model     = 'dpt-2-latest',
    save_to   = GROUNDING_DIR,
)
elapsed_ade  = round(time.time() - t0, 2)
chunks       = ade_response.chunks
markdown_ade = ade_response.markdown or ''
print(f'  Chunks   : {len(chunks)}')
print(f'  MD chars : {len(markdown_ade):,}')
print(f'  Elapsed  : {elapsed_ade}s')


In [0]:
# ── Raw chunk inspection ─────────────────────────────────────────────────
# Prints each chunk's full repr, useful for understanding the response
# structure before building downstream logic.
#
# Each Chunk exposes:
#   chunk.id          : unique UUID
#   chunk.type        : element type string  (text / table / marginalia / ...)
#   chunk.markdown    : content as Markdown or HTML
#   chunk.grounding   : ChunkGrounding(page, box)
#     .page           : 0-based page index
#     .box            : ParseGroundingBox(top, bottom, left, right)  -- normalised [0,1]
for chunk in chunks:
    print(chunk)

In [0]:
import base64, io, re
import fitz
from PIL import Image
import markdown as md_lib

CHUNK_COLOURS = {
    "text"       : "#3B82F6",
    "table"      : "#F97316",
    "marginalia" : "#9CA3AF",
    "figure"     : "#10B981",
    "header"     : "#8B5CF6",
    "caption"    : "#EAB308",
    "list"       : "#14B8A6",
    "formula"    : "#F43F5E",
}
DEFAULT_COLOUR = "#6B7280"


def _pil_to_b64(pil_img: Image.Image, max_width: int = 900) -> str:
    if pil_img.width > max_width:
        ratio   = max_width / pil_img.width
        pil_img = pil_img.resize((max_width, int(pil_img.height * ratio)), Image.LANCZOS)
    buf = io.BytesIO()
    pil_img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def _clean_markdown(md: str) -> str:
    """Supprime les ancres <a id='...'></a> injectees par LandingAI."""
    return re.sub(r"<a id='[^']*'></a>\n\n*", "", md or "").strip()


def render_landingai_output(
    chunks,
    pdf_path: str,
    title: str = "LandingAI ADE - Debug Visualisation",
    dpi: int = 150,
):
    doc = fitz.open(pdf_path)
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    page_images = []
    for page in doc:
        pix = page.get_pixmap(matrix=mat, alpha=False)
        page_images.append(Image.frombytes("RGB", [pix.width, pix.height], pix.samples))
    doc.close()
    print(f"{len(page_images)} page(s) rasterised")

    pages_html = ""
    for idx, pil_img in enumerate(page_images):
        b64         = _pil_to_b64(pil_img)
        chunks_html = ""
        for c in chunks:
            chunk_type = str(getattr(c, "type", "text")).split(".")[-1].lower()
            colour     = CHUNK_COLOURS.get(chunk_type, DEFAULT_COLOUR)
            clean_md   = _clean_markdown(c.markdown)
            rendered   = md_lib.markdown(clean_md, extensions=["tables", "extra"])
            chunks_html += (
                f'<div style="margin-bottom:12px;padding:10px 12px;'
                f'border-left:3px solid {colour};background:#FAFAFA;'
                f'border-radius:0 6px 6px 0;font-size:13px;line-height:1.6;">'
                f'<span style="font-size:10px;font-weight:700;color:{colour};'
                f'text-transform:uppercase;letter-spacing:.06em;">{chunk_type}</span>'
                f'<div style="margin-top:6px;color:#1F2937;overflow-x:auto;">{rendered}</div>'
                f'</div>'
            )
        if not chunks_html:
            chunks_html = '<p style="color:#9CA3AF;font-size:13px;">No chunks on this page.</p>'
        pages_html += (
            f'<div style="margin-bottom:48px;">'
            f'<h3 style="margin:0 0 12px;font-size:15px;color:#374151;">Page {idx + 1} '
            f'<span style="font-weight:400;font-size:12px;color:#9CA3AF;">{len(chunks)} chunk(s)</span></h3>'
            f'<div style="display:flex;gap:24px;align-items:flex-start;">'
            f'<div style="flex:0 0 auto;"><img src="{b64}" '
            f'style="display:block;max-width:600px;width:100%;border:1px solid #E5E7EB;border-radius:6px;"/></div>'
            f'<div style="flex:1;min-width:0;">{chunks_html}</div>'
            f'</div></div>'
        )

    displayHTML(
        f'<div style="font-family:Inter,ui-sans-serif,system-ui,sans-serif;'
        f'max-width:1400px;padding:24px 32px;background:#FFFFFF;'
        f'border:1px solid #E5E7EB;border-radius:10px;margin:16px 0;">'
        f'<h2 style="margin:0 0 16px;font-size:18px;color:#111827;">{title}</h2>'
        f'{pages_html}</div>'
    )

In [0]:
render_landingai_output(
    chunks   = chunks,
    pdf_path = f"{PATH_VOLUME}/AccidentStatement.pdf",   # ou PREPROCESSED_PDF
    title    = "LandingAI ADE — AccidentStatement.pdf",
)